In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
Tesla T4


In [ ]:
!pip install ultralytics supervision opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 8.9 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
results = model.predict(
    source="video.mp4",
    save=True,
    conf=0.4
)

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



KeyboardInterrupt: 

In [ ]:
model.track(
    source="video.mp4",
    save=True,
    conf=0.4
)

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



In [ ]:
from ultralytics import YOLO
import supervision as sv
import numpy as np

# Initialiser le modèle YOLO
model = YOLO("yolov8n.pt")

# Définir les points de départ et d'arrivée de la ligne
# Ces coordonnées sont des exemples. Vous devrez les ajuster en fonction
# de la position de la ligne que vous souhaitez surveiller dans votre vidéo.
# Les coordonnées sont (x, y)
line_start = sv.Point(500, 500) # Exemple: point (x,y) de départ de la ligne
line_end = sv.Point(1500, 500)   # Exemple: point (x,y) d'arrivée de la ligne

# Créer un détecteur de ligne
line_zone = sv.LineZone(start=line_start, end=line_end)

# Initialiser un annotateur pour dessiner la ligne et le compteur
# Suppression de 'text_thickness' et 'text_scale' car ils ne sont plus supportés par certaines versions de supervision
line_zone_annotator = sv.LineZoneAnnotator(thickness=2)

# Initialiser un annotateur pour les boîtes de détection et les identifiants de suivi
# Suppression de 'text_thickness' et 'text_scale' car ils ne sont plus supportés par certaines versions de supervision
box_annotator = sv.BoxAnnotator(thickness=2)

# Initialiser un annotateur pour les étiquettes (labels)
label_annotator = sv.LabelAnnotator()

# Maintenant, nous devons re-exécuter le modèle avec 'stream=True' pour traiter chaque frame
# et appliquer la logique de la ligne. Les résultats précédents étaient un objet unique.
# Avec stream=True, nous obtenons un générateur de résultats, image par image.

print("Début du traitement de la vidéo avec détection de ligne...")

# Créer un écrivain de vidéo pour sauvegarder les résultats avec les annotations
# Le nom du fichier de sortie sera 'video_output_with_line_tracked.mp4'
video_info = sv.VideoInfo.from_video_path("video.mp4")
with sv.VideoSink(target_path="video_output_with_line_tracked.mp4", video_info=video_info) as sink:
    for result in model.track(source="video.mp4", stream=True, conf=0.4, verbose=False, tracker="bytetrack.yaml"):
        frame = result.orig_img
        detections = sv.Detections.from_ultralytics(result)

        # Mettre à jour la zone de ligne avec les nouvelles détections
        # Appeler line_zone.trigger uniquement si des tracker_id sont présents
        if detections.tracker_id is not None and len(detections.tracker_id) > 0:
            # Filtrer les détections pour s'assurer que seuls celles avec un tracker_id non-None sont passées
            # Bien que tracker_id soit souvent un np.ndarray, il peut contenir des None si le tracker échoue
            # pour certains objets. La méthode from_ultralytics devrait normalement le gérer.
            # Si detections.tracker_id est un tableau d'entiers, cela suffit.
            # Si c'est un mélange d'entiers et de None, une étape de filtrage plus complexe serait nécessaire.
            # Pour l'instant, on suppose que s'il n'est pas None, il contient des IDs valides pour supervision.
            line_zone.trigger(detections=detections)

        # Initialiser annotated_frame avec la frame originale
        annotated_frame = frame.copy()

        # Créer les étiquettes à afficher
        labels = []
        if len(detections) > 0:
            if detections.tracker_id is not None: # Si des IDs de suivi sont disponibles
                labels = [
                    f"#{tracker_id} {model.names[class_id]} {confidence:0.2f}"
                    for class_id, tracker_id, confidence
                    in zip(detections.class_id, detections.tracker_id, detections.confidence)
                ]
            elif detections.class_id is not None and detections.confidence is not None:
                # Si pas d'IDs de suivi, mais des détections, créer des étiquettes sans ID
                labels = [
                    f"{model.names[class_id]} {confidence:0.2f}"
                    for class_id, confidence
                    in zip(detections.class_id, detections.confidence)
                ]

            # Annoter le cadre avec les boîtes
            annotated_frame = box_annotator.annotate(
                scene=annotated_frame, detections=detections
            )
            # Annoter les étiquettes séparément
            annotated_frame = label_annotator.annotate(
                scene=annotated_frame, detections=detections, labels=labels
            )

        # Annoter la ligne de zone, même s'il n'y a pas de détections
        annotated_frame = line_zone_annotator.annotate(
            frame=annotated_frame, line_counter=line_zone
        )

        sink.write_frame(frame=annotated_frame)

print("Traitement terminé. La vidéo annotée est sauvegardée sous 'video_output_with_line_tracked.mp4'.")
print(f"Entrées: {line_zone.in_count}, Sorties: {line_zone.out_count}")

Début du traitement de la vidéo avec détection de ligne...
Traitement terminé. La vidéo annotée est sauvegardée sous 'video_output_with_line_tracked.mp4'.
Entrées: 1, Sorties: 0
